# Patron de Reflexion: un agente que se autocritica

Este notebook muestra el patron de **reflexion** (generar -> criticar -> revisar): un LLM produce una respuesta, otro rol del mismo modelo (el "revisor") la critica, y si encuentra un problema el modelo la corrige. Se repite hasta que el revisor la aprueba o se agotan los intentos.

Caso de uso: un agente que explica conceptos de fisica a un estudiante, y se autocorrige antes de entregar la respuesta final.

Las dos "voces" (explicador y revisor) son el mismo LLM con distintas instrucciones de sistema.

Requisitos: Ollama corriendo localmente con al menos el modelo `llama3.2:1b` descargado (ver `ayuda.txt`).

In [1]:
from reflection_agent import explicar, revisar, corregir, run_reflection
from display_helpers import show_trace

## 1. Sin reflexion (linea base)

Primero pedimos la explicacion una sola vez, sin ningun paso de revision. Con un modelo pequeno como `llama3.2:1b`, a veces la explicacion tiene errores o imprecisiones que pasan sin que nadie las revise.

In [3]:
concepto = "por que el periodo de un pendulo simple no depende de la masa"
model = "llama3.2:1b"

explicacion_base = explicar(concepto, model)
print(explicacion_base)

¡Claro! En un pendúlo simple, la masa del pendú y el objeto que se está subiendo no tiene ningún efecto directo en el periodo de descenso. El periodo es la duración en la que se repite el movimiento de descenso del pendú.

En realidad, el periodo de un pendú simple depende de dos factores fundamentales:

* La constante de gravitación de la Tierra (g): es la fuerza que actúa sobre el pendú y el objeto. Si aumentamos la masa del pendú o el objeto, el valor de g también aumenta.
* La pendiente de la pendículo: es la razón entre la longitud del pendículo y su altura. Una pendiente más grande significa que el pendú se mueve más rápidamente por una cantidad más grande de altura.

En cuanto a la masa, no hay relación directa con el periodo de descenso. Lo que importa es la relación entre la masa del pendú y el objeto y la pendiente de la pendículo.


## 2. El ciclo paso a paso

El patron tiene tres funciones en `reflection_agent.py`:

- `explicar(concepto, model)` -- genera una primera explicacion.
- `revisar(concepto, explicacion, model)` -- el mismo modelo, con otra instruccion de sistema, actua como revisor: dice si la aprueba y por que no si la rechaza.
- `corregir(concepto, explicacion, comentarios, model)` -- vuelve a generar la explicacion, esta vez viendo los comentarios del revisor.

Vamos a correrlas a mano una vez, sobre la explicacion de la seccion anterior.

In [4]:
revision = revisar(concepto, explicacion_base, model)
print("Aprobada:", revision["aprobada"])
print(revision["comentarios"])

Aprobada: False
APROBADA
El concepto de periodo de un pendú simple es correcto, pero la explicación del profesor no es clara en algunos aspectos. La relación entre la masa y el periodo es directa, ya que la masa del pendú y el objeto influyen en la fuerza de la gravedad que actúa sobre el pendú. Por ejemplo, un pendú más pesado tendrá un periodo mayor. Sin embargo, no mencionó la relación entre la masa y la pendiente, lo que es fundamental para entender por qué el periodo no depende de la masa.


In [5]:
if not revision["aprobada"]:
    explicacion_corregida = corregir(concepto, explicacion_base, revision["comentarios"], model)
    print(explicacion_corregida)
else:
    print("El revisor la aprobo, no hace falta corregir.")

¡Muchas gracias por la corrección! Aquí te presento una versión corregida de la explicación:

El concepto de periodo de un pendú simple es correcto, pero la explicación del profesor no es clara en algunos aspectos. La relación entre la masa y el periodo es directa, ya que la masa del pendú y el objeto influye en la fuerza de la gravedad que actúa sobre el pendú. Por ejemplo, un pendú más pesado tendrá un periodo mayor. Sin embargo, la relación entre la masa y la pendiente es más importante para entender por qué el periodo no depende de la masa.

La pendiente de la pendículo es la razón entre la longitud del pendículo y su altura. Un pendículo más largo o más estrecho significa que el pendú se mueve más rápidamente por una cantidad más grande de altura. Por lo tanto, una pendiente más grande tendrá un periodo mayor. En otras palabras, la pendiente influye directamente en el periodo de descenso.


## 3. El ciclo completo: `run_reflection`

`run_reflection` encadena estos tres pasos automaticamente hasta `max_rounds` veces, o hasta que el revisor aprueba. Devuelve la traza completa (cada ronda con su explicacion y su revision) para poder inspeccionar que paso.

In [6]:
trace = run_reflection(concepto, model=model, max_rounds=3)
show_trace(concepto, trace)

## Concepto: por que el periodo de un pendulo simple no depende de la masa

### Ronda 0

¡Claro! El período de un pendulo simple es una cuestión interesante.

En general, el período de un pendulo simple es la duración que tarda en caer una bola a lo largo de un eje vertical. Ahora, ¿por qué no depende de la masa? La respuesta es que la masa no es un factor significativo en este caso.

En realidad, la masa de la bola no tiene mucho efecto en el período del pendulo. La fuerza de la gravedad es la que impulsa la bola a caer, y esta fuerza depende del peso de la bola, que es proporcional a la masa. Sin embargo, la velocidad a la que la bola cae también es importante.

La velocidad a la que una bola cae es determinada por la altura de la pendiente y la masa de la bola. Si la pendiente es demasiado alta, la bola caerá más rápidamente. Pero la masa de la bola no afecta esta velocidad.

La razón es que la masa de la bola no es una función de la velocidad. En otras palabras, la masa es una función de la masa misma, no de la velocidad. Por lo tanto, la masa no puede afectar la velocidad a la que la bola cae, y por lo tanto, no afecta el período del pendulo.

En resumen, la masa de la bola no es un factor significativo en el período del pendulo simple, y la velocidad a la que la bola cae es la única variable que afecta la duración del período.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA
Comentario: El profesor tiene una explicación clara y concisa, pero debería mencionar la diferencia entre el período de un pendulo simple y el período de un pendulo de mass point. Además, en algunos pasos, se mencionan que la masa no afecta la velocidad, pero no se explica claramente cómo.

### Ronda 1

...

### Ronda 2

¡Muchas gracias por la retroalimentación! Aquí te dejo la versión corregida completa:

 Explico: por qué el período de un pendulo simple no depende de la masa

En general, el período de un pendulo simple es la duración que tarda en caer una bola a lo largo de un eje vertical. Pero, ¿por qué no depende de la masa? La respuesta es que la masa no es un factor significativo en este caso.

En realidad, la masa de la bola no tiene mucho efecto en el período del pendulo. La fuerza de la gravedad es la que impulsa la bola a caer, y esta fuerza depende del peso de la bola, que es proporcional a la masa. Sin embargo, la velocidad a la que la bola cae también es importante.

La velocidad a la que una bola cae es determinada por la altura de la pendiente y la masa de la bola. Si la pendiente es demasiado alta, la bola caerá más rápidamente. Pero la masa de la bola no afecta esta velocidad.

La razón es que la masa de la bola no es una función lineal de la velocidad. En otras palabras, la masa es una función polinómica: la masa de un mass point es una función de su masa, no de su velocidad. Esto significa que la masa no puede reducirse a una función lineal de la velocidad.

En otras palabras, la velocidad a la que una bola cae no es una función lineal de su masa, sino que es una función polinómica. Por lo tanto, la masa no puede afectar la velocidad a la que cayó.

En el caso de un pendulo simple, un mass point (un punto de masa) cayendo desde una altura no tiene un período significativo. El período es simplemente la inversa del acelerador: el período es la distancia que recorre la bola, que depende solo de la masa, la velocidad inicial y la gravedad. En otras palabras, la masa de un mass point no afecta la velocidad a la que cayó, pero sí afecta el período del pendulo.

En resumen, la masa de la bola no es un factor significativo en el período del pendulo simple, y la velocidad a la que la bola cae es la única variable que afecta la duración del período. La masa de un mass point, por otro lado, tiene un período significativo, pero solo depende de la masa, la velocidad inicial y la gravedad.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA

COMENTARIOS: la respuesta se ajusta al concepto, pero la explicación se puede mejorar un poco. Para ser aún más claro, podrías haber expresado que la masa de un mass point es una función polinómica: la masa de un mass point es un polinomio de su masa.

### Ronda 3

¡Muchas gracias por la retroalimentación! Aquí te dejo la versión corregida completa:

Explico: por qué el período de un pendulo simple no depende de la masa

En general, el período de un pendulo simple es la duración que tarda en caer una bola a lo largo de un eje vertical. Pero, ¿por qué no depende de la masa? La respuesta es que la masa de un mass point es una función polinómica: la masa de un mass point es un polinomio de su masa. La función polinómica la describe de la siguiente manera: la masa de un mass point es igual a mi masa multiplicada por un factor constante, que depende de su posición y momento. En el caso de un pendulo simple, la masa de la bola es la misma en todas las posiciones.

En realidad, la masa de la bola no tiene mucha influencia en el período del pendulo. La fuerza de la gravedad es la que impulsa la bola a caer, y esta fuerza depende del peso de la bola, que es proporcional a su masa. Sin embargo, la velocidad a la que la bola cae también es importante. La velocidad a la que una bola cae es determinada por la altura de la pendiente y la masa de la bola. Si la pendiente es demasiado alta, la bola caerá más rápidamente. Pero la masa de la bola no afecta esta velocidad.

La razón es que la masa de la bola no es una función lineal de la velocidad. En otras palabras, la masa de un mass point es una función polinómica: la masa de un mass point es igual a mi masa multiplicada por un factor constante, que depende de su posición y momento. Esto significa que la masa no puede reducirse a una función lineal de la velocidad.

En el caso de un pendulo simple, un mass point (un punto de masa) cayendo desde una altura no tiene un período significativo. El período es simplemente la inversa del acelerador: el período es la distancia que recorre la bola, que depende solo de su masa, la velocidad inicial y la gravedad. En otras palabras, la masa de un mass point no afecta la velocidad a la que cayó, pero sí afecta el período del pendulo.

En resumen, la masa de un mass point es una función polinómica de su masa, la masa no tiene un período significativo en un pendulo simple, y la masa de un mass point no afecta la velocidad a la que cayó, pero sí afecta el período del pendulo.

## 4. Varios conceptos

Probemos con un par de conceptos mas -- el codigo no cambia, solo el texto que le pasamos.

In [7]:
conceptos = [
    "por que un objeto en caida libre y otro lanzado horizontalmente desde la misma altura tocan el suelo al mismo tiempo",
    "diferencia entre velocidad y aceleracion",
]

for c in conceptos:
    trace = run_reflection(c, model=model, max_rounds=3)
    show_trace(c, trace)

## Concepto: por que un objeto en caida libre y otro lanzado horizontalmente desde la misma altura tocan el suelo al mismo tiempo

### Ronda 0

¡Claro! Vamos a analizar el problema paso a paso.

Imagina que dos objetos están en caída libre desde la misma altura. El primer objeto, una pelota de béisbol, se cae verticalmente hacia abajo con una velocidad constante. El segundo objeto, una pelota de béisbol, se mueve horizontalmente desde la misma altura con una velocidad constante, pero en sentido contrario al cañón de caída.

En la caída de la pelota de béisbol, la fuerza de atracción gravitatoria entre la pelota y el suelo es constante y opuesta a la velocidad de caída. Sin embargo, la pelota de béisbol se mueve horizontalmente, lo que significa que la fuerza de atracción gravitatoria también es constante, pero en sentido contrario a la velocidad de caída.

En la caída de la pelota de béisbol horizontalmente, la velocidad de caída es igual a la velocidad tangencial del objeto (la velocidad que se mide en ángulo recto con el eje de caída). La fuerza de atracción gravitatoria es igual a la masa del objeto multiplicada por la gravedad (masa = m x g).

Dado que la velocidad de caída es constante para ambas pelotas, la fuerza de atracción gravitatoria también es constante para ambas. Sin embargo, la masa del objeto es diferente. La masa de la pelota de béisbol es mayor que la masa de la pelota de béisbol lanzada horizontalmente.

Por lo tanto, la pelota de béisbol horizontalmente lanza una velocidad mayor que la pelota de béisbol verticalmente debido a su mayor masa. Esto significa que la pelota de béisbol horizontalmente recibe más energía potencial de la pelota de béisbol verticalmente y se mueve más rápido a su caída final.

En resumen, la velocidad tangencial de los dos objetos es la misma, pero la masa del objeto es mayor para la pelota de béisbol horizontalmente. Esto provoca que la pelota de béisbol horizontalmente recibe más energía potencial de la pelota de béisbol verticalmente y se mueve más rápido a su caída final.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA

COMENTARIOS: La explicación del profesor es clara y concisa, pero podría ser mejor explicar con más detalle la ecuación que establece la diferencia de masa entre los dos objetos. Por ejemplo, podrías mencionar que la masa de la pelota de béisbol horizontalmente es m + M, donde m es la masa del objeto, y M es la masa de la pelota de béisbol verticalmente.

### Ronda 1

...

### Ronda 2

¡Muchas gracias por la corrección! Me alegra saber que mi explicación original era efectiva y que la revisión ayudó a mejorarla. Aquí te dejo la versión corregida:

La velocidad y la aceleración se refieren a conceptos diferentes. La velocidad es una medida de la distancia que un objeto cubre en un tiempo determinado, mientras que la aceleración es la velocidad a la que un objeto se mueve cambiando de velocidad.

En otras palabras, la velocidad es un valor numérico que mide la distancia que un objeto cubre en un tiempo determinado, mientras que la aceleración es el sentido en el que se mide, como la dirección de un vector. La aceleración es positiva, pero es un sentido en el que se mide, como la dirección de un vector.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA

COMENTARIOS: La respuesta original es efectiva, pero la explicación podría ser más clara y concisa. Por ejemplo, en lugar de decir "en un tiempo determinado", podrías decir "en un período de tiempo determinado". Además, la segunda oración podría ser más concisa y enfocada en la definición de aceleración en el sentido de la dirección de un vector.

### Ronda 3

Me agradecer por la corrección y el feedback. Aquí te dejo la versión corregida:

La velocidad y la aceleración se refieren a conceptos diferentes. La velocidad es una medida de la distancia que un objeto cubre en un período de tiempo determinado, mientras que la aceleración es la velocidad a la que un objeto se mueve cambiando de velocidad.

En otras palabras, la velocidad es un valor numérico que mide la distancia que un objeto cubre en un período de tiempo determinado, mientras que la aceleración es el sentido en el que se mide, como la dirección de un vector. La aceleración es positiva, pero es un sentido en el que se mide.

## 5. Modelo pequeno vs modelo mas grande

`llama3.2:1b` es rapido pero poco confiable, y eso se nota **en los dos roles**: el explicador puede cometer errores, y el revisor puede no detectarlos (o rechazar explicaciones que si estaban bien). Un modelo mas grande, como `qwen3:1.7b`, suele ser mas consistente en ambos roles -- sin tocar el resto del codigo, solo cambia el parametro `model`.

In [8]:
model_grande = "qwen3:1.7b"
trace = run_reflection(concepto, model=model_grande, max_rounds=3)
show_trace(concepto, trace)

## Concepto: por que el periodo de un pendulo simple no depende de la masa

### Ronda 0

El periodo de un péndulo simple no depende de la masa porque la fórmula del período, $ T = 2\pi \sqrt{\frac{L}{g}} $, muestra que la masa se cancela en el cálculo. Aunque la masa participa en la segunda ley de Newton ($ F = ma $), en el caso del péndulo, la aceleración es proporcionada por la gravedad, y su valor no depende de la masa. Además, la masa no influye en el momento de inercia del péndulo, que es $ I = mL^2 $, y así se cancela en la expresión del período. Por lo tanto, el periodo solo depende del largo del péndulo y de la aceleración de la gravedad, no de la masa.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA:  
COMENTARIOS: La explicación es correcta y clara, destacando que la masa se cancela en la fórmula del período y en el momento de inercia. No hay errores conceptuales o imprecisiones.

### Ronda 1

...

### Ronda 2

La explicación es correcta y clara, pero se puede mejorar para ser más precisa al destacar que la masa se cancela en la derivación de la fórmula del período debido al uso del momento de inercia, que depende proporcionalmente de la masa. Aunque la masa participa en la segunda ley de Newton, en el caso del péndulo, el momento de inercia se expresa como $ I = mL^2 $, donde $ L $ es el largo del hilo. Al sustituir este en la fórmula del período $ T = 2\pi \sqrt{\frac{L}{g}} $, se observa que la masa se anula, ya que $ L $ es constante y $ g $ también lo es. Esto demuestra que el período de un péndulo simple depende solo de su largo y de la aceleración de la gravedad, no de la masa. La precisión de esta observación es clave para entender el comportamiento físico del péndulo.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA:  
COMENTARIOS: La explicación es correcta y clara, pero se puede mejorar para ser más precisa al destacar que la masa se anula debido al uso del momento de inercia $ I = mL^2 $, que se substituye en la fórmula del período $ T = 2\pi \sqrt{\frac{L}{g}} $, eliminando la masa. La precisión de esta observación es clave, aunque el mencionar la segunda ley de Newton no es necesaria para el razonamiento.

### Ronda 3

La explicación es correcta y clara, pero se puede mejorar para ser más precisa al destacar que la masa se anula debido al uso del momento de inercia $ I = mL^2 $, que se sustituye en la fórmula del período $ T = 2\pi \sqrt{\frac{L}{g}} $. Al reemplazar $ I $ por $ mL^2 $, se observa que la masa se cancela, ya que $ L $ y $ g $ son constantes. Esto demuestra que el período de un péndulo simple depende solo de su largo y de la aceleración de la gravedad, no de la masa. La precisión de esta observación es clave, aunque el mencionar la segunda ley de Newton no es necesaria para el razonamiento.


La idea central -- **un LLM que critica y corrige su propia salida antes de entregarla** -- se traslada a otros flujos de un curso o laboratorio:

- **Retroalimentacion de tareas**: el agente redacta un comentario sobre la respuesta de un estudiante, y un segundo paso revisa que la retroalimentacion sea justa y este bien fundamentada antes de mostrarsela.
- **Otros conceptos**: solo cambia el texto que le pasas a `run_reflection`, no hace falta tocar el codigo.
- **Otro criterio de revision**: cambia `REVISOR_PROMPT` en `reflection_agent.py` para que el revisor chequee otra cosa (ej. que la explicacion use el vocabulario correcto para el nivel del curso, o que incluya una analogia).
- **Combinarlo con herramientas**: se puede mezclar con el patron de `Tools` -- por ejemplo, un agente que ajusta una curva y *despues* reflexiona sobre si el modelo elegido tiene sentido fisico.